# Homework 4 Submission

Moving beyond intuition to empirically compare keyword, vector, and hybrid search methods, generating a ground-truth dataset from 72 standardized course lessons (commit `8c1834d`), then evaluate and rank these search techniques using concrete performance metrics.

## Imports and Setup

In [1]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from google import genai
from pydantic import BaseModel
from typing import List
from tqdm.auto import tqdm

from gitsource import GithubRepositoryDataReader, chunk_documents
from embedder import Embedder
import minsearch

load_dotenv('../hw1/.env')
client = genai.Client()

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

## Q1: Generating Ground Truth Data
What's the average number of input tokens across these 3 calls?

In [2]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

class Questions(BaseModel):
    questions: List[str]

total_input_tokens = 0
for idx, path in enumerate([
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md"
]):
    doc = next(d for d in documents if d["filename"] == path)
    user_prompt = f"filename: {doc['filename']}\ncontent: {doc['content']}"
    
    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=user_prompt,
        config=genai.types.GenerateContentConfig(
            system_instruction=data_gen_instructions,
            response_mime_type="application/json",
            response_schema=Questions,
            temperature=0.0
        )
    )
    total_input_tokens += response.usage_metadata.prompt_token_count

print(f"Q1 Average input tokens: {total_input_tokens / 3:.2f}")

Q1 Average input tokens: 1294.67


## Q2: Text Search
After running text_search for it, what's the filename of the first result?

In [8]:
ground_truth = pd.read_csv('ground-truth.csv').to_dict(orient='records')
chunks = chunk_documents(documents, size=2000, step=1000)

tindex = minsearch.Index(text_fields=["content"], keyword_fields=["filename"])
tindex.fit(chunks)

def text_search(query, num_results=5):
    return tindex.search(query, num_results=num_results)

q1_query = ground_truth[0]["question"]
text_res = text_search(q1_query, num_results=5)
print(f"Q2 Text search first filename: {text_res[0]['filename']}")

Q2 Text search first filename: 01-agentic-rag/lessons/03-rag.md


## Q3: Vector Search
After running vector_search for the same question, what's the filename of the first result?

In [4]:
embed = Embedder()
chunk_contents = [chunk["content"] for chunk in chunks]
X = embed.encode_batch(chunk_contents)

vindex = minsearch.VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

def vector_search(query, num_results=5):
    v_q = embed.encode(query)
    return vindex.search(v_q, num_results=num_results)

vec_res = vector_search(q1_query, num_results=5)
print(f"Q3 Vector search first filename: {vec_res[0]['filename']}")

Q3 Vector search first filename: 01-agentic-rag/lessons/01-intro.md


## Q4: Evaluating Text Search
What's the Hit Rate?

In [5]:
def compute_relevance(search_results, target_filename):
    return [1 if doc["filename"] == target_filename else 0 for doc in search_results]

def hit_rate(relevance_total):
    cnt = 0
    for line in relevance_total:
        if 1 in line:
            cnt += 1
    return cnt / len(relevance_total)

def mrr(relevance_total):
    total_score = 0.0
    for line in relevance_total:
        for rank, val in enumerate(line):
            if val == 1:
                total_score += 1 / (rank + 1)
                break
    return total_score / len(relevance_total)

def evaluate(search_function, data, num_results=5):
    relevance_total = []
    for row in tqdm(data):
        q = row["question"]
        doc = row["filename"]
        results = search_function(q, num_results=num_results)
        relevance = compute_relevance(results, doc)
        relevance_total.append(relevance)
    return hit_rate(relevance_total), mrr(relevance_total)

hr_text, mrr_text = evaluate(text_search, ground_truth)
print(f"Q4 Text search hit rate: {hr_text:.4f}")

  0%|          | 0/360 [00:00<?, ?it/s]

Q4 Text search hit rate: 0.7583


## Q5: Evaluating Vector Search
What's the MRR?

In [6]:
hr_vec, mrr_vec = evaluate(vector_search, ground_truth)
print(f"Q5 Vector search MRR: {mrr_vec:.4f}")

  0%|          | 0/360 [00:00<?, ?it/s]

Q5 Vector search MRR: 0.5486


## Q6: Evaluating Hybrid Search
Which k gives the best MRR?

In [7]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

def hybrid_search(query, k=60, num_results=5):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k, num_results=num_results)

def evaluate_hybrid(data, k):
    relevance_total = []
    for row in data:
        q = row["question"]
        doc = row["filename"]
        results = hybrid_search(q, k=k)
        relevance = compute_relevance(results, doc)
        relevance_total.append(relevance)
    return hit_rate(relevance_total), mrr(relevance_total)

for k in [1, 50, 100, 200]:
    hr, m = evaluate_hybrid(ground_truth, k)
    print(f"k={k}: MRR={m:.4f}")

k=1: MRR=0.6482
k=50: MRR=0.6379
k=100: MRR=0.6379
k=200: MRR=0.6379
